In [2]:
import pandas as pd
import duckdb
from pathlib import Path

PROJECT_ROOT = Path.cwd()

PARQUET_DIR = (
    PROJECT_ROOT / "data" / "processed" / "ratings_parquet"
)

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "splits"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = OUTPUT_DIR / "train.parquet"
TEST_PATH = OUTPUT_DIR / "test.parquet"

con = duckdb.connect()

con.execute("SET memory_limit = '2GB'")
con.execute("SET threads = 2")

In [3]:
ratings_path = PARQUET_DIR.as_posix()

query = f"""
SELECT
    userId,
    movieId,
    rating,
    timestamp,
    CASE
        WHEN rating_count >= 2 AND row_num = 1
        THEN 'test'
        ELSE 'train'
    END AS split

FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY userId
            ORDER BY timestamp DESC, movieId DESC
        ) AS row_num,

        COUNT(*) OVER (
            PARTITION BY userId
        ) AS rating_count

    FROM read_parquet('{ratings_path}/*.parquet')
)
"""

split_query = f"""
COPY ({query})
TO '{(OUTPUT_DIR / "ratings_split.parquet").as_posix()}'
(FORMAT PARQUET, COMPRESSION ZSTD);
"""

con.execute(split_query)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
split_path = (OUTPUT_DIR / "ratings_split.parquet").as_posix()

con.execute(f"""
SELECT
    split,
    COUNT(*) AS total_ratings,
    COUNT(DISTINCT userId) AS unique_users
FROM read_parquet('{split_path}')
GROUP BY split
""").df()

,split,total_ratings,unique_users
0,test,138493,138493
1,train,19861770,138493


In [6]:
con.execute(f"""
SELECT DISTINCT userId
FROM read_parquet('{split_path}')

EXCEPT

SELECT DISTINCT userId
FROM read_parquet('{split_path}')
WHERE split = 'train'
""").df()

,userId


In [7]:
TRAIN_PATH = OUTPUT_DIR / "train.parquet"
TEST_PATH = OUTPUT_DIR / "test.parquet"

SPLIT_PATH = OUTPUT_DIR / "ratings_split.parquet"

In [8]:
con.execute(f"""
COPY (
    SELECT userId, movieId, rating, timestamp
    FROM read_parquet('{SPLIT_PATH.as_posix()}')
    WHERE split = 'train'
)
TO '{TRAIN_PATH.as_posix()}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [9]:
con.execute(f"""
COPY (
    SELECT userId, movieId, rating, timestamp
    FROM read_parquet('{SPLIT_PATH.as_posix()}')
    WHERE split = 'test'
)
TO '{TEST_PATH.as_posix()}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [10]:
train_summary = con.execute(f"""
SELECT
    COUNT(*) AS total_ratings,
    COUNT(DISTINCT userId) AS unique_users,
    COUNT(DISTINCT movieId) AS unique_movies
FROM read_parquet('{TRAIN_PATH.as_posix()}')
""").df()

test_summary = con.execute(f"""
SELECT
    COUNT(*) AS total_ratings,
    COUNT(DISTINCT userId) AS unique_users,
    COUNT(DISTINCT movieId) AS unique_movies
FROM read_parquet('{TEST_PATH.as_posix()}')
""").df()

print("TRAIN DATA")
display(train_summary)

print("TEST DATA")
display(test_summary)

TRAIN DATA


,total_ratings,unique_users,unique_movies
0,19861770,138493,26707


TEST DATA


,total_ratings,unique_users,unique_movies
0,138493,138493,9405


In [11]:
con.execute(f"""
SELECT COUNT(*) AS overlapping_interactions
FROM read_parquet('{TRAIN_PATH.as_posix()}') train
INNER JOIN read_parquet('{TEST_PATH.as_posix()}') test
ON train.userId = test.userId
AND train.movieId = test.movieId
""").df()

,overlapping_interactions
0,0


In [12]:
display(train_summary)
display(test_summary)

,total_ratings,unique_users,unique_movies
0,19861770,138493,26707


,total_ratings,unique_users,unique_movies
0,138493,138493,9405
